# Module 10: Data Visualization — Solutions

**Datasets:** Iris, Titanic  
**Objective:** Complete solutions for all visualization exercises

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from matplotlib.gridspec import GridSpec

%matplotlib inline
sns.set_theme()

iris_data = load_iris()
iris = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
iris['species'] = iris_data.target_names[iris_data.target]

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)
print('Datasets loaded.')

### Solution 1: Basic Matplotlib (OO API)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter
species_colors = {'setosa': 'red', 'versicolor': 'green', 'virginica': 'blue'}
for sp in iris['species'].unique():
    subset = iris[iris['species'] == sp]
    axes[0].scatter(subset['sepal length (cm)'], subset['petal length (cm)'],
                   c=species_colors[sp], label=sp, alpha=0.7, s=50)
axes[0].set_xlabel('Sepal Length (cm)')
axes[0].set_ylabel('Petal Length (cm)')
axes[0].set_title('Sepal vs Petal Length')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: line
axes[1].plot(range(50), iris['sepal length (cm)'][:50], 'b-o', label='Sepal Length', alpha=0.7)
axes[1].plot(range(50), iris['sepal width (cm)'][:50], 'r-s', label='Sepal Width', alpha=0.7)
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Length (cm)')
axes[1].set_title('First 50 Samples')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Solution 2: Histograms & Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
n, bins, patches = axes[0].hist(titanic['Age'].dropna(), bins=25, density=True,
                                color='steelblue', edgecolor='black', alpha=0.7)
from scipy.stats import gaussian_kde
age_clean = titanic['Age'].dropna()
kde = gaussian_kde(age_clean)
x_grid = np.linspace(0, 80, 200)
axes[0].plot(x_grid, kde(x_grid), 'r-', linewidth=2, label='KDE')
axes[0].axvline(age_clean.mean(), color='red', linestyle='--', label=f'Mean: {age_clean.mean():.1f}')
axes[0].axvline(age_clean.median(), color='green', linestyle=':', label=f'Median: {age_clean.median():.1f}')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Density')
axes[0].set_title('Age Distribution with KDE')
axes[0].legend()

# Overlapping histogram by survival
axes[1].hist(titanic.loc[titanic['Survived'] == 1, 'Age'].dropna(), bins=25,
             alpha=0.5, color='green', label='Survived', density=True)
axes[1].hist(titanic.loc[titanic['Survived'] == 0, 'Age'].dropna(), bins=25,
             alpha=0.5, color='red', label='Not Survived', density=True)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Density')
axes[1].set_title('Age Distribution by Survival')
axes[1].legend()

plt.tight_layout()
plt.show()

### Solution 3: Box Plots & Outliers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

titanic.boxplot(column='Age', by='Pclass', ax=axes[0, 0])
axes[0, 0].set_title('Age by Class')

titanic.boxplot(column='Fare', by='Embarked', ax=axes[0, 1])
axes[0, 1].set_title('Fare by Embarkation Port')

# IQR outlier count
def count_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return ((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum()

outlier_counts = titanic.groupby('Pclass')['Fare'].apply(count_outliers)
axes[1, 0].bar(outlier_counts.index, outlier_counts.values, color=['gold', 'silver', '#CD7F32'])
axes[1, 0].set_title('Fare Outlier Count by Class')
axes[1, 0].set_xlabel('Pclass')
axes[1, 0].set_ylabel('Outlier Count')

# Horizontal box plot
axes[1, 1].boxplot([titanic[titanic['Pclass'] == i]['Fare'].dropna() for i in [1, 2, 3]],
                   labels=[1, 2, 3], vert=False, patch_artist=True)
axes[1, 1].set_title('Fare by Class (Horizontal)')
axes[1, 1].set_xlabel('Fare')

plt.tight_layout()
plt.show()

### Solution 4: Correlation Heatmap

In [ ]:
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr = titanic[numeric_cols].corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = corr.copy()
corr_masked.values[mask] = np.nan

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_masked, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        if not mask[i, j]:
            val = corr.iloc[i, j]
            color = 'white' if abs(val) > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=11)

plt.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Titanic Correlation Matrix (Lower Triangle)')
plt.tight_layout()
plt.show()

### Solution 5: Seaborn Pairplot & Jointplot

In [ ]:
sns.pairplot(iris, hue='species', diag_kind='hist', palette='Set2')
plt.suptitle('Iris Pairplot', y=1.02)
plt.show()

sns.jointplot(data=iris, x='petal length (cm)', y='petal width (cm)',
              hue='species', kind='kde')
plt.suptitle('Petal Length vs Width (KDE)', y=1.02)
plt.show()

sns.jointplot(data=titanic, x='Age', y='Fare', kind='hex', height=8)
plt.suptitle('Age vs Fare (Hexbin)', y=1.02)
plt.show()

print('Setosa is the easiest to separate — petal features work best.')

### Solution 6: Seaborn Categorical Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=titanic, x='Sex', hue='Survived', ax=axes[0, 0])
axes[0, 0].set_title('Survival by Sex')

sns.barplot(data=titanic, x='Pclass', y='Survived', ax=axes[0, 1])
axes[0, 1].set_title('Survival Rate by Class')

sns.violinplot(data=titanic, x='Pclass', y='Age', hue='Survived', split=True, ax=axes[1, 0])
axes[1, 0].set_title('Age by Class and Survival')

sns.catplot(data=titanic, col='Sex', x='Pclass', y='Survived', kind='point', height=4)
plt.suptitle('Survival by Class and Sex', y=1.05)
plt.show()

### Solution 7: lmplot & Custom Regression

In [ ]:
sns.lmplot(data=titanic, x='Age', y='Fare', hue='Survived',
           scatter_kws={'alpha': 0.4}, height=6)
plt.suptitle('Fare vs Age by Survival', y=1.02)
plt.show()

sns.lmplot(data=iris, x='petal length (cm)', y='petal width (cm)',
           hue='species', order=2, scatter_kws={'alpha': 0.6}, height=6)
plt.suptitle('Polynomial Regression: Petal Features', y=1.02)
plt.show()

### Solution 8: Complex Subplots with GridSpec

In [ ]:
fig = plt.figure(figsize=(15, 12))
gs = GridSpec(3, 3, figure=fig, height_ratios=[1, 1.5, 1])

# Top row
ax0 = fig.add_subplot(gs[0, 0])
ax0.hist(titanic['Age'].dropna(), bins=25, color='steelblue', edgecolor='white')
ax0.set_title('Age Distribution')

ax1 = fig.add_subplot(gs[0, 1])
ax1.hist(titanic['Fare'], bins=40, color='coral', edgecolor='white')
ax1.set_title('Fare Distribution')

ax2 = fig.add_subplot(gs[0, 2])
titanic.boxplot(column='Age', by='Pclass', ax=ax2)
ax2.set_title('Age by Pclass')

# Middle row (spanning all columns)
ax3 = fig.add_subplot(gs[1, :])
ax3.scatter(titanic['Age'], titanic['Fare'], c=titanic['Survived'],
            cmap='RdYlGn', alpha=0.5, edgecolors='gray', linewidth=0.3)
ax3.set_xlabel('Age')
ax3.set_ylabel('Fare')
ax3.set_title('Age vs Fare (Green=Survived, Red=Not)')

# Bottom row
ax4 = fig.add_subplot(gs[2, 0])
titanic['Sex'].value_counts().plot(kind='bar', ax=ax4, color=['pink', 'blue'])
ax4.set_title('Sex Count')

ax5 = fig.add_subplot(gs[2, 1])
titanic['Embarked'].value_counts().plot(kind='bar', ax=ax5, color='purple')
ax5.set_title('Embarked Count')

ax6 = fig.add_subplot(gs[2, 2])
titanic['Pclass'].value_counts().sort_index().plot(kind='pie', ax=ax6, autopct='%1.1f%%',
                                                    colors=['gold', 'silver', '#CD7F32'])
ax6.set_title('Pclass Distribution')
ax6.set_ylabel('')

plt.tight_layout()
plt.show()

### Solution 9: Boxen Plot & Advanced Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxenplot(data=titanic, x='Pclass', y='Fare', ax=axes[0, 0])
axes[0, 0].set_title('Boxen Plot: Fare by Class')

sns.boxplot(data=titanic, x='Pclass', y='Fare', ax=axes[0, 1])
axes[0, 1].set_title('Box Plot: Fare by Class')

sns.violinplot(data=titanic, x='Sex', y='Age', hue='Survived', split=True, ax=axes[1, 0])
axes[1, 0].set_title('Age by Sex and Survival')

sns.ecdfplot(data=titanic, x='Age', hue='Survived', ax=axes[1, 1])
axes[1, 1].set_title('ECDF: Age by Survival')

plt.tight_layout()
plt.show()

### Solution 10: Full EDA Visualization Report

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 3, figure=fig)

# 1. Correlation heatmap
ax0 = fig.add_subplot(gs[0, 0])
corr = titanic[numeric_cols].corr()
im = ax0.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax0.set_xticks(range(len(numeric_cols)))
ax0.set_yticks(range(len(numeric_cols)))
ax0.set_xticklabels(numeric_cols, rotation=45, ha='right', fontsize=7)
ax0.set_yticklabels(numeric_cols, fontsize=7)
ax0.set_title('Correlation Matrix', fontsize=10)

# 2. Survival counts
ax1 = fig.add_subplot(gs[0, 1])
titanic['Survived'].value_counts().plot(kind='bar', ax=ax1, color=['#E74C3C', '#2ECC71'])
ax1.set_xticklabels(['No', 'Yes'])
ax1.set_title('Survival (38% survived)', fontsize=10)

# 3. Age distribution by survival
ax2 = fig.add_subplot(gs[0, 2])
sns.histplot(data=titanic, x='Age', hue='Survived', bins=25, alpha=0.5, ax=ax2)
ax2.set_title('Age by Survival', fontsize=10)

# 4-6. Row 2: Class, Gender, Fare
ax3 = fig.add_subplot(gs[1, 0])
sns.barplot(data=titanic, x='Pclass', y='Survived', ax=ax3)
ax3.set_title('Survival by Class', fontsize=10)

ax4 = fig.add_subplot(gs[1, 1])
sns.barplot(data=titanic, x='Sex', y='Survived', ax=ax4)
ax4.set_title('Survival by Gender', fontsize=10)

ax5 = fig.add_subplot(gs[1, 2])
sns.boxenplot(data=titanic, x='Pclass', y='Fare', hue='Survived', ax=ax5)
ax5.set_title('Fare by Class & Survival', fontsize=10)

# 7-9. Row 3: Embarked, Age scatter, Family
ax6 = fig.add_subplot(gs[2, 0])
sns.countplot(data=titanic, x='Embarked', hue='Survived', ax=ax6)
ax6.set_title('Survival by Embarkation', fontsize=10)

ax7 = fig.add_subplot(gs[2, 1])
ax7.scatter(titanic['Age'], titanic['Fare'], c=titanic['Survived'],
            cmap='RdYlGn', alpha=0.4, s=20)
ax7.set_xlabel('Age')
ax7.set_ylabel('Fare')
ax7.set_title('Age vs Fare', fontsize=10)

ax8 = fig.add_subplot(gs[2, 2])
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
sns.barplot(data=titanic, x='FamilySize', y='Survived', ax=ax8)
ax8.set_title('Survival by Family Size', fontsize=10)

plt.suptitle('Titanic Dataset: EDA Report for ML', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print('\n=== ML-Relevant Insights ===')
print('1. Pclass and Sex are the strongest categorical predictors of survival')
print('2. Age shows bimodal distribution; children have higher survival')
print('3. Fare is highly skewed; log-transform may help models')
print('4. Family size of 2-4 correlates with higher survival probability')